In [53]:
import torch
torch.set_default_dtype(torch.float64)
from cpu_cb import E8P12_codebook

In [54]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k) / (2**k)

In [55]:
def calculate_k(n):
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    return k

In [56]:
def hb_transform_loop(x, k): # hb_transform without the end permutate and reshape
    (m,n) = x.shape
    assert 4**k == n
    b = torch.tensor([1, 1, -1, -1]).to(x.dtype)
    x = x.reshape((m,) + (4,) * k)
    for i in range(k):
        x = x.flip(1 + i) + x * b.view((1,) * (i + 1) + (4,) + (1,) * (k - 1 - i))
    # return x.reshape((m,) + (2, 2) * k) / (2**k)
    return x.reshape((m,) + (2, 2) * k)

In [57]:
def hb_transform_reshape(x, k): # given original hb_transform function shape go to the hb_transform_loop
    m = x.shape[0]
    x = x.reshape((m,) + (2,) * (2 * k))
    forward_perm = [0] + [2 * i + 1 for i in range(k)] + [2 * i + 2 for i in range(k)]
    inverse_perm = [0] * (2 * k + 1)
    for i, p in enumerate(forward_perm):
        inverse_perm[p] = i 
    x = x.permute(inverse_perm)
    return x.reshape((m,) + (4,) * k)

In [58]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]
cb = E8P12_codebook()

In [59]:
def bt(M, n):
    k = calculate_k(n * n)
    M_flat = hb_transform_reshape(M.reshape(1, n, n), k).reshape(1, n * n)
    result = hb_transform_loop(M_flat, k) / (2**k)
    return result.reshape(n * n) / (1.0 / n)

def ibt(error, n):
    return hb_transform(error.reshape(1, n * n)).reshape(n, n)

In [60]:
def decompose_H(H_sqrt, n):
    decomp = []
    while n > 8:
        next_n = n // 2
        Ha, Hb = H_sqrt[:next_n, :next_n], H_sqrt[:next_n, next_n:]
        Hc, Hd = H_sqrt[next_n:, :next_n], H_sqrt[next_n:, next_n:]
        H_decomp = [(Ha+Hd)/2, (Hb+Hc)/2, (Hb-Hc)/2, (Ha-Hd)/2]
        decomp.append((H_sqrt, H_decomp))
        H_sqrt = H_decomp[0]
        n = next_n
    decomp.append((H_sqrt, None))
    return decomp

In [61]:
B_2 = hb_transform(torch.eye(4)) * 2
cached_signs = {}
for b in range(4):
    for a in range(b):
        for i in range(4):
            if torch.allclose(B_2[i] @ B_2[a], B_2[b]):
                cached_signs[(a, b)] = (i, 1)
            if torch.allclose(B_2[i] @ B_2[a], -B_2[b]):
                cached_signs[(a, b)] = (i, -1)
# print(cached_signs)
basis_mats = hb_transform(torch.eye(64)) * 8    

In [62]:
def fast_orth_quant(coeffs, n, hat_coeffs, error, decomp, iter):
    H_sqrt, H_decomp = decomp[iter]
    tr_H_sqrt = torch.diagonal(H_sqrt).sum()
    if n == 8:
        # J = torch.zeros(64, 64, dtype=H_sqrt.dtype, device=H_sqrt.device)
        # for c in range(8):
        #     B_curr = basis_mats[cliques[c]]
        #     for cp in range(c):
        #         B_prior = basis_mats[cliques[cp]]
        #         block = ((B_curr @ H_sqrt).unsqueeze(1) * B_prior.unsqueeze(0)).sum(dim=(-2, -1)) / tr_H_sqrt / 64
        #         for i, r in enumerate(cliques[c]):
        #             for j, cl in enumerate(cliques[cp]):
        #                 J[r, cl] = block[i, j]
        J = ((basis_mats @ H_sqrt).unsqueeze(1) * basis_mats.unsqueeze(0)).sum(dim=(-2,-1)) / tr_H_sqrt / 64
        for c in range(8):
            target = coeffs[cliques[c]] + J[cliques[c]] @ error
            hat_coeffs[cliques[c]] = cb.quantize(target.unsqueeze(0))[0].squeeze(0)
            error[cliques[c]] = coeffs[cliques[c]] - hat_coeffs[cliques[c]]
        return
    next_n = n // 2
    cliques_per_group = (n * n) // 32
    group_size = (n * n) // 4
    clique_vals = torch.stack([torch.tensor(cliques[c % 8], dtype=torch.long) + 64 * (c // 8) for c in range(cliques_per_group)])
    for b in range(4):
        corrections = torch.zeros(cliques_per_group, 8, dtype=coeffs.dtype)
        for a in range(b):
            d, sign = cached_signs[(a, b)]
            e_a = error[a * group_size:(a+1) * group_size]
            f = bt(ibt(e_a, next_n) @ H_decomp[d], next_n)
            corrections += sign * f[clique_vals] / tr_H_sqrt / next_n**2
        coeffs_b = coeffs[b * group_size:(b+1) * group_size].clone()
        for c_ind in range(cliques_per_group):
            coeffs_b[clique_vals[c_ind]] += corrections[c_ind]
        hat_b = hat_coeffs[b * group_size:(b+1) * group_size]
        error_b = error[b * group_size:(b+1) * group_size]
        fast_orth_quant(coeffs_b, next_n, hat_b, error_b, decomp, iter + 1)

In [63]:
n = 64
k = calculate_k(n * n)
# U_W, _ = torch.linalg.qr(torch.randn(n, n))
# V_W, _ = torch.linalg.qr(torch.randn(n, n))
# lambda_W = torch.tensor([1.0 / i**2 for i in range(1, n+1)], dtype=torch.float64)
# W = U_W @ torch.diag(lambda_W) @ V_W.T
W = torch.randn(n, n)
# U_H, _ = torch.linalg.qr(torch.randn(n, n))
# lambda_H = torch.tensor([1.0 / i for i in range(1, n+1)], dtype=torch.float64)
# H = U_H @ torch.diag(lambda_H) @ U_H.T
H = torch.randn(n,n)
H = H @ H.t()
H = (H + H.t())/2
H.diagonal().add_(H.diagonal().mean() * 0.01)

tensor([97.2610, 69.2744, 49.1006, 67.4815, 92.2549, 79.4517, 52.6458, 84.2287,
        73.8559, 59.8079, 51.6349, 54.6729, 46.9962, 67.5286, 54.7065, 72.1365,
        49.2111, 69.1009, 74.9400, 45.1705, 82.0198, 91.9558, 56.2389, 86.8838,
        49.8031, 60.4151, 51.9784, 60.3296, 81.3419, 51.7884, 71.0112, 68.0327,
        64.9499, 53.7254, 56.4775, 66.5050, 57.2420, 82.0103, 57.3511, 61.1165,
        45.0324, 68.9144, 66.6797, 69.9289, 58.8016, 67.5710, 53.9507, 41.3677,
        72.2053, 88.9374, 71.3149, 82.4309, 60.1955, 63.4197, 84.8494, 60.6034,
        54.0876, 42.2794, 61.7201, 75.5711, 60.2102, 74.2120, 71.7778, 58.2597])

In [64]:
print("Adaptive rounding")
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
error = torch.zeros(n * n, dtype=coeffs.dtype)
decomp = decompose_H(H_sqrt, n)
fast_orth_quant(coeffs, n, hat_coeffs, error, decomp, 0)
hat_coeffs = hat_coeffs * Wscale
hatW_fast = hb_transform(hat_coeffs.reshape(1, n*n)).reshape(n, n)
err = (W - hatW_fast).norm() / W.norm()
print(f"Error: {err}")
proxy_loss = ((hatW_fast - W) @ H @ (hatW_fast - W).T).trace()
print(f"Proxy loss: {proxy_loss}")

Adaptive rounding
Error: 0.30076317204190894
Proxy loss: 23394.117533298377


In [65]:
print("No adaptive rounding")
H_nr = torch.eye(n)
S, V = torch.linalg.eigh(H_nr)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = torch.zeros_like(coeffs)
error = torch.zeros(n * n, dtype=coeffs.dtype)
decomp = decompose_H(H_sqrt, n)
fast_orth_quant(coeffs, n, hat_coeffs, error, decomp, 0)
hat_coeffs = hat_coeffs * Wscale
hatW_fast = hb_transform(hat_coeffs.reshape(1, n*n)).reshape(n, n)
err = (W - hatW_fast).norm() / W.norm()
print(f"Error: {err}")
proxy_loss = ((hatW_fast - W) @ H @ (hatW_fast - W).T).trace()
print(f"Proxy loss: {proxy_loss}")

No adaptive rounding
Error: 0.30076296429591204
Proxy loss: 23497.662984366187
